# Evaluation pipeline demo


This is a short notebook showing how the evaluation pipeline works WITHOUT having to download the entire dataset and configure the container. Small subsets of each dataset is included in duckdb files in this folder.

Ensure you have Python 3 and `uv`, then just run `uv sync` from the repository root and ensure the notebook is connected to that virtual environment (likely `.venv` by default)

## Datasets
1. [Risk-Based Authentication (RBA)](https://zenodo.org/records/6782156) (Wiefling et al, 2022)
    - synthetic dataset of 31m authentication records from ~3m users with user IDs from an SSO service
    - includes user agent, IP, etc., as well as fields for (a) whether the login originates from a known attack IP address or (b) the login was flagged as an account takeover (ground truth)
    - Licensed by the authors (Wiefling et al.) under a [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

2. [FP-Stalker](https://github.com/Spirals-Team/FPStalker) (Vastel et al., 2018) 
    - 15k record sample from the AmIUnique dataset from 2015-2017. 
    - Each record contains a long term tracking cookie (tying it to other records from the same origin) alongside a full user agent string and other fingerprint attributes we do not use in our ER method (e.g., canvas).
    - Licensed by the authors (Vastel et al.) under the [GNU-AGPL 3.0 License](https://github.com/Spirals-Team/FPStalker/blob/master/LICENSE)



The datasets provided in this demo are small subsets of the above datasets just to show how this works at a high level:
1. RBA subset (3.2k rows): Selected only only "User IDs" that have at least one row flagged as "Is Account Takeover"
2. FP Stalker Subset (8.6k rows): Selected only tracking cookie IDs that were live in June 2016. Also removed some columns that aren't useful to us.


In [15]:
import duckdb
import pandas as pd

with duckdb.connect('./trunc_rba.duckdb') as rba_conn:
    rba_df_orig = rba_conn.execute("SELECT * FROM imported_data").df()

print("Length of RBA truncated dataset:", rba_df_orig.shape[0])
display(rba_df_orig.head())

Length of RBA truncated dataset: 3242


,index,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,OS Name and Version,Device Type,Login Successful,Is Attack IP,Is Account Takeover
0,136248,2020-02-05 08:05:59.945,9022188698077912916,NaN,92.220.83.153,NO,Agder,Staubo,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 11_2_6 lik...,Firefox 20.0.0.1618,iOS 11.2.6,mobile,True,False,False
1,136449,2020-02-05 08:09:03.208,9022188698077912916,NaN,92.220.83.153,NO,Agder,Staubo,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 11_2_6 lik...,Chrome Mobile WebView 85.0.4183,iOS 11.2.6,mobile,True,False,False
2,141104,2020-02-05 09:16:56.531,1453346545260446038,NaN,79.161.43.238,NO,Rogaland,Karmsund,29695,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,iOS 7.1,mobile,True,False,False
3,144281,2020-02-05 10:00:09.843,7168556392326159106,NaN,10.0.77.214,BR,-,-,29695,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6...,Chrome 69.0.3497.17.28,Mac OS X 10.14.6,desktop,True,False,False
4,152021,2020-02-05 11:51:32.626,6650085773542423553,NaN,10.2.59.250,BR,Paraíba,Frei Martinho,500194,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6...,Chrome 86.0.4202,Mac OS X 10.14.6,desktop,True,False,False


In [16]:
with duckdb.connect('./trunc_fp_stalker.duckdb') as fp_conn:
    fp_df_orig = fp_conn.execute("SELECT * FROM imported_data").df() 

print("Length of FP Stalker truncated dataset:", fp_df_orig.shape[0])
display(fp_df_orig.head())

Length of FP Stalker truncated dataset: 8646


,counter,id,creationDate,updateDate,endDate,userAgentHttp,osDetailed,browserDetailed,browserVersion,platformJS,resolutionJS
0,1,f4f25187-82c7-4265-9bf9-71bc8ed1701f,2015-10-15 14:00:00,2015-10-16 14:00:00,2015-10-16 19:00:00,Mozilla/5.0 (Windows NT 6.1; WOW64; rv:41.0) G...,Windows 7,Firefox,,Win32,1920x1200x24
1,8,d4be492d-c9ee-46a0-b1ab-076386b61b85,2015-10-16 13:00:00,2015-10-19 13:00:00,2015-10-20 09:00:00,Mozilla/5.0 (Macintosh; Intel Mac OS X 10.10; ...,Mac OS X,Firefox,,MacIntel,1920x1080x24
2,9,f66e4b87-4a5b-498d-a2d1-6a35f2310ff2,2015-10-16 19:00:00,2015-10-18 14:00:00,2015-10-20 05:00:00,Mozilla/5.0 (Macintosh; Intel Mac OS X 10.10; ...,,,,MacIntel,1280x800x24
3,10,10c196fc-2c76-4e4b-b628-4472949c39c3,2015-10-16 21:00:00,NaT,2015-10-19 07:00:00,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,,,,Linux x86_64,1600x900x24
4,11,bc19c32d-a271-4dd7-b5e6-62d36d1a3128,2015-10-16 21:00:00,2015-10-17 12:00:00,2015-10-17 16:00:00,Mozilla/5.0 (Windows NT 6.3; WOW64; rv:41.0) G...,Windows 8.1,Firefox,,Win32,1600x900x24


# Parse User Agent Strings 

Parse the user agent strings (e.g., `Mozilla/5.0 (Windows NT 6.1; WOW64; rv:41.0) Gecko/20100101 Firefox/41.0`) using the same `field_normalization` modules as the webapp, then move them into the column names expected by the `device_grouping2` module.

In [17]:
import sys
from pathlib import Path
import tqdm

_root = Path.cwd().parent.parent
_python_core = str(_root / "python_core")
if _python_core not in sys.path:
    sys.path.append(_python_core)
if str(_root) not in sys.path:
    sys.path.append(str(_root))

from python_core.field_normalization.user_agent import UserAgentParser
from python_core.field_normalization.device import normalize_device_fields

In [18]:
ua_parser = UserAgentParser()

def normalize_data(df: pd.DataFrame, 
                   user_agent_col_name: str, 
                   timestamp_col_name: str,
                   other_cols_to_keep: list = []) -> pd.DataFrame:
    dicts = []
    for _, row in tqdm.tqdm(df.iterrows(), total=len(df)):
        d = {"user_agent_original": row.get(user_agent_col_name, "")}
        d = ua_parser.parse(d) # returns dict
        d = normalize_device_fields(d)
        d = {k: v for k, v in d.items() if k.startswith("norm__")}
        d['timestamp'] = row.get(timestamp_col_name, "")
        dicts.append(d)
    
    new_columns = pd.DataFrame(dicts)
    normalized_cols = [c for c in new_columns.columns if c.startswith("norm__")]
    
    if isinstance(other_cols_to_keep, list):
        other_cols_to_keep = [c for c in other_cols_to_keep if c in df.columns]
    else:
        other_cols_to_keep = []
    
    new_df = df[other_cols_to_keep + [user_agent_col_name]]
    new_df = pd.concat([
        new_columns[["timestamp"] + normalized_cols], 
        new_df], axis=1)
    new_df.rename(columns={c: f"attr__{c}" for c in normalized_cols}, inplace=True)
    return new_df

In [19]:
s = "Mozilla/5.0  (iPhone; CPU iPhone OS 11_2_6 like Mac OS X) AppleWebKit/537.36 (KHTML, like Gecko Version/4.0 Chrome/85.0.4183.127 Mobile Safari/537.36Snapchat11.1.7.81 (LYA-L29; Android 10#10.1.0.288C432#29; gzip"
s = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/69.0.3497.17.19.92 Safari/537.36"
s = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.1 Safari/605.1.15"
s = "Mozilla/5.0  (iPad; CPU OS 7_1 like Mac OS X) AppleWebKit/533.1 (KHTML, like Gecko Version/4.0 Mobile Safari/533.1 variation/277457"
ua_parser.parse({"user_agent_original": s})

{'user_agent_client_name': 'Mobile Safari',
 'user_agent_client_version': '4.0',
 'user_agent_client_type': <AppType.Browser: 'browser'>,
 'user_agent_secondary_client_name': 'variation',
 'user_agent_secondary_client_version': '277457',
 'user_agent_secondary_client_type': <AppType.Generic: 'generic'>,
 'user_agent_is_mobile': True,
 'user_agent_uses_mobile_browser': True,
 'user_agent_os_name': 'iOS',
 'user_agent_os_type': re.compile(r'ios|ipados', re.IGNORECASE|re.UNICODE),
 'user_agent_os_version': '7.1',
 'user_agent_device_model_name': 'iPad',
 'user_agent_device_manufacturer': 'Apple',
 'user_agent_device_type': 'tablet'}

In [20]:
norm_rba_df = normalize_data(rba_df_orig,
                             user_agent_col_name='User Agent String', 
                             timestamp_col_name='Login Timestamp',
                             other_cols_to_keep=['index', 'User ID', 'Login Successful', 'Is Account Takeover', 'Is Attack IP', 'Browser Name and Version', 'OS Name and Version'])                           

norm_rba_df.rename(columns={'index': 'id'},  inplace=True)  # "id" has to be the unique id of the row
norm_rba_df.convert_dtypes()
norm_rba_df

100%|██████████| 3242/3242 [00:03<00:00, 824.97it/s] 


,timestamp,attr__norm__model_name,attr__norm__manufacturer,attr__norm__os_name,attr__norm__os_version,attr__norm__os_type,attr__norm__client_name,attr__norm__client_version,id,User ID,Login Successful,Is Account Takeover,Is Attack IP,Browser Name and Version,OS Name and Version,User Agent String
0,2020-02-05 08:05:59.945,iphone,apple,ios,11.2.6,ios,firefox,20.0.0.1618,136248,9022188698077912916,True,False,False,Firefox 20.0.0.1618,iOS 11.2.6,Mozilla/5.0 (iPhone; CPU iPhone OS 11_2_6 lik...
1,2020-02-05 08:09:03.208,iphone,apple,android,10,android,snapchat,11.1.7.81,136449,9022188698077912916,True,False,False,Chrome Mobile WebView 85.0.4183,iOS 11.2.6,Mozilla/5.0 (iPhone; CPU iPhone OS 11_2_6 lik...
2,2020-02-05 09:16:56.531,ipad,apple,ios,7.1,ios,mobile safari :: variation,4.0,141104,1453346545260446038,True,False,False,Android 2.3.3.2672,iOS 7.1,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...
3,2020-02-05 10:00:09.843,macintosh,apple,mac,10.14.6,macos,chrome,69.0.3497.17.28.100,144281,7168556392326159106,True,False,False,Chrome 69.0.3497.17.28,Mac OS X 10.14.6,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6...
4,2020-02-05 11:51:32.626,macintosh,apple,mac,10.14.6,macos,chrome,86.0.4202.0,152021,6650085773542423553,True,False,False,Chrome 86.0.4202,Mac OS X 10.14.6,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3237,2021-02-16 15:14:45.233,NaN,NaN,chrome os,12607.82.0,linux,chrome,79.0.3945.192.204.123,30309533,3713430081263795573,True,False,False,Chrome 79.0.3945.192.204,Chrome OS 12607.82.0,Mozilla/5.0 (X11; CrOS x86_64 12607.82.0) Appl...
3238,2021-02-16 15:25:46.491,NaN,NaN,chrome os,12607.82.0,linux,chrome,79.0.3945.192.204.123,30310725,3713430081263795573,True,False,False,Chrome 79.0.3945.192.204,Chrome OS 12607.82.0,Mozilla/5.0 (X11; CrOS x86_64 12607.82.0) Appl...
3239,2021-02-16 15:30:31.066,NaN,NaN,chrome os,12607.82.0,linux,chrome,79.0.3945.192.204.123,30311205,3713430081263795573,True,False,False,Chrome 79.0.3945.192.204,Chrome OS 12607.82.0,Mozilla/5.0 (X11; CrOS x86_64 12607.82.0) Appl...
3240,2021-02-26 16:34:12.232,macintosh,apple,mac,10.14.6,macos,chrome :: variation,69.0.3497.17.24.100,31094961,7470558100220665624,True,False,True,Chrome 69.0.3497.17.24,Mac OS X 10.14.6,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6...


In [21]:
norm_fp_df = normalize_data(fp_df_orig,
                            user_agent_col_name='userAgentHttp', 
                            timestamp_col_name='creationDate',
                            other_cols_to_keep=['id', 'counter', 'osDetailed', 'browserDetailed'])

norm_fp_df.rename(columns={'id': 'tracking_id'},  inplace=True)  # "id" has to be the unique id of the row
norm_fp_df.rename(columns={'counter': 'id'}, inplace=True)

norm_fp_df.convert_dtypes()
norm_fp_df

100%|██████████| 8646/8646 [00:07<00:00, 1093.70it/s]


,timestamp,attr__norm__os_name,attr__norm__os_version,attr__norm__os_type,attr__norm__client_name,attr__norm__client_version,attr__norm__model_name,attr__norm__manufacturer,tracking_id,id,osDetailed,browserDetailed,userAgentHttp
0,2015-10-15 14:00:00,windows,7,windows,firefox,41.0,NaN,NaN,f4f25187-82c7-4265-9bf9-71bc8ed1701f,1,Windows 7,Firefox,Mozilla/5.0 (Windows NT 6.1; WOW64; rv:41.0) G...
1,2015-10-16 13:00:00,mac,10.10,macos,firefox,41.0,macintosh,apple,d4be492d-c9ee-46a0-b1ab-076386b61b85,8,Mac OS X,Firefox,Mozilla/5.0 (Macintosh; Intel Mac OS X 10.10; ...
2,2015-10-16 19:00:00,mac,10.10,macos,firefox,41.0,macintosh,apple,f66e4b87-4a5b-498d-a2d1-6a35f2310ff2,9,,,Mozilla/5.0 (Macintosh; Intel Mac OS X 10.10; ...
3,2015-10-16 21:00:00,linux,NaN,linux,chrome,46.0.2490.71,NaN,NaN,10c196fc-2c76-4e4b-b628-4472949c39c3,10,,,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...
4,2015-10-16 21:00:00,windows,8.1,windows,firefox,41.0,NaN,NaN,bc19c32d-a271-4dd7-b5e6-62d36d1a3128,11,Windows 8.1,Firefox,Mozilla/5.0 (Windows NT 6.3; WOW64; rv:41.0) G...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8641,2016-08-31 07:00:00,linux,NaN,linux,chrome,52.0.2743.116,NaN,NaN,10c196fc-2c76-4e4b-b628-4472949c39c3,14991,,,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...
8642,2016-08-31 09:00:00,windows,10,windows,chrome,52.0.2743.116,NaN,NaN,2f38656b-76d5-4b46-a81d-c90b96495673,14996,Windows 10,Chrome,Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebK...
8643,2016-08-31 09:00:00,mac,10.10.3,macos,chrome,42.0.2311.152,macintosh,apple,fe23c50e-5966-4307-a5c9-3aaaf4d0b4b7,14997,,,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_3...
8644,2016-08-31 09:00:00,mac,NaN,macos,midori,0.4,macintosh,apple,fe23c50e-5966-4307-a5c9-3aaaf4d0b4b7,14999,,,Mozilla/5.0 (Macintosh; U; Intel Mac OS X; en-...


# something

In [22]:
from python_core.device_grouping2 import client_os_upgrades, profiles
from python_core.device_grouping2.instances import DeviceInstanceGraph
import datetime
import json

In [36]:
start_time  = datetime.datetime.now().isoformat()

rba_results = {
    "run": {
        "description": "Truncated RBA dataset for demo",
        "start_time": start_time,
    },
    "summary": {},
    "computed": []
}

# -----------

for user_id in map(int, norm_rba_df['User ID'].unique()):
    user_df = norm_rba_df[norm_rba_df['User ID'] == user_id]

    row_ids_with_account_takeover = set(user_df[user_df['Is Account Takeover'] == True]['id'].tolist())
    row_ids_with_attack_ip = set(user_df[user_df['Is Attack IP'] == True]['id'].tolist())

    upgrade_edges = client_os_upgrades.get_edges(user_df) # build up edges between records
    graph = DeviceInstanceGraph(user_df, pd.DataFrame(upgrade_edges)) # construct subgraphs from all the edges

    for i, inst in enumerate(graph.get_instances()):
        dct = {}
        dct["user_id"] = user_id
        dct["instance_number"] = i

        dct["instance_metadata"] = inst.export_as_dict()
        
        records_with_account_takeover = list(row_ids_with_account_takeover.intersection(set(inst.df['id'].tolist())))
        records_with_attack_ip = list(row_ids_with_attack_ip.intersection(set(inst.df['id'].tolist())))
        
        dct['record_ids'] = {
            "all": inst.df['id'].tolist(), # each subgraph is an "instance"
            "account_takeover": records_with_account_takeover,
            "attack_ip": records_with_attack_ip,
            "both_account_takeover_and_attack_ip": list(set(records_with_account_takeover).intersection(set(records_with_attack_ip))),
            "benign": list(set(inst.df['id'].tolist()).difference(set(records_with_account_takeover)).difference(set(records_with_attack_ip))),
        }

        dct['summary'] = {
            "num_records": len(dct['record_ids']),
            "num_records_with_account_takeover": len(records_with_account_takeover),
            "num_records_with_attack_ip": len(records_with_attack_ip),
            "num_records_with_both_account_takeover_and_attack_ip": len(dct['record_ids']['both_account_takeover_and_attack_ip']),
            "num_records_benign": len(dct['record_ids']['benign']),
        }

        rba_results["computed"].append(dct)


# -----------

rba_results["run"]["end_time"] = datetime.datetime.now().isoformat()
rba_results["run"]["duration"] = str(datetime.datetime.fromisoformat(rba_results["run"]["end_time"]) - datetime.datetime.fromisoformat(start_time))


In [40]:
# compute summary 
num_singleton = 0
num_takeover_inst = 0
num_only_benign = 0
num_takeover_and_benign = 0
num_attack_ip_inst = 0
num_attack_ip_and_benign = 0
sizes = []
takeover_and_benign_instances = []

for inst in rba_results["computed"]:
    r_ids = inst['record_ids']
    sz = len(r_ids['all'])
    sizes.append(sz)
    if sz == 1: num_singleton += 1
        
    has_takeover = len(r_ids['account_takeover']) > 0
    has_attack = len(r_ids['attack_ip']) > 0
    has_benign = len(r_ids['benign']) > 0
    
    if has_takeover:
        num_takeover_inst += 1
        if has_benign:
            num_takeover_and_benign += 1
            takeover_and_benign_instances.append(inst)

    if has_attack:
        num_attack_ip_inst += 1
        if has_benign:
            num_attack_ip_and_benign += 1
    if has_benign and not has_takeover and not has_attack:
        num_only_benign += 1

s_sizes = pd.Series(sizes) if sizes else pd.Series([0])

rba_results["summary"] = {
    "num_records": len(norm_rba_df),
    "num_users": len(norm_rba_df['User ID'].unique()),
    "num_instances": len(rba_results["computed"]),
    "instance_sizes": {
        "num_singleton_instances": num_singleton,
        "median_instance_size": int(s_sizes.median()),
        "mean_instance_size": float(s_sizes.mean()),
        "max_instance_size": int(s_sizes.max()),
    },
    "account_takeover_stats": {
        "num_records_with_account_takeover": int(norm_rba_df['Is Account Takeover'].sum()),
        "num_instances_with_account_takeover": num_takeover_inst,
        "num_instances_only_benign": num_only_benign,
        "num_instances_both_takeover_and_benign": num_takeover_and_benign,
    },
    "attack_ip_stats": {
        "num_records_with_attack_ip": int(norm_rba_df['Is Attack IP'].sum()),
        "num_instances_with_attack_ip": num_attack_ip_inst,
        "num_instances_both_attack_ip_and_benign": num_attack_ip_and_benign,
    }
}

In [41]:
# make dir /rundata
_start_time = start_time.replace(":", "-").replace(".", "-")
dirpath =  Path("runs") / f"rba_{_start_time}"
dirpath.mkdir(parents=True, exist_ok=True)
with open(dirpath / "rba_results.json", "w") as f:
    json.dump(rba_results, f, indent=4)
norm_rba_df.to_csv(dirpath / "rba_results.csv", index=False)

with open(dirpath / "takeout_and_benign_instances.json", "w") as f:
    json.dump(takeover_and_benign_instances, f, indent=4)
